How it works
- Block structure: Each block stores an index, timestamp, data, previous_hash, and nonce. The block’s hash is computed from these fields, making it tamper‑evident.
- Linking: previous_hash connects each block to the one before it—changing any block breaks the chain’s continuity.
- Hashing: A block’s hash is the SHA‑256 digest of a deterministic JSON serialization of its contents.
- Proof‑of‑work: The miner increments nonce until the hash has difficulty leading zeros, making it costly to rewrite history.
- Validation: is_chain_valid() rechecks links, recomputes hashes, and verifies proof‑of‑work for every block.


1. What is a Blockchain?
- A blockchain is like a digital notebook where each page (block) is linked to the previous one.
- Each block contains:
- Data (like transactions)
- Timestamp (when it was created)
- Hash (a unique fingerprint of the block)
- Previous Hash (fingerprint of the block before it)
- This linking makes it secure: if someone changes one block, all following blocks break.

2. Code Overview
The program has three main parts:
- Hashing function – creates unique fingerprints.
- Block class – defines what a block looks like.
- Blockchain class – manages the chain of blocks.




In [ ]:
import hashlib
import json
import time

3. Hashing Function

In [3]:
def sha256(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

- Purpose: Converts any text into a fixed 64‑character string.
- Why: Hashing ensures data integrity. Even a small change in input gives a completely different hash.
- Example: "hello" → 2cf24dba5fb0...




4. Block Class

In [16]:
class Block:
    def __init__(self, index: int, timestamp: float, data: dict, previous_hash: str, nonce: int = 0):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        # Deterministic serialization for hashing
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True, separators=(",", ":"))
        return sha256(block_string)

Key Attributes:
- index → Position in the chain (0 = first block).
- timestamp → When the block was created.
- data → Information stored (e.g., transactions).
- previous_hash → Links this block to the one before.
- nonce → A number used for mining (proof‑of‑work).
- hash → Unique fingerprint of the block.
Hash Calculation


Hash Calculation

- Converts block data into a JSON string.
- Hashes it with SHA‑256.
- Ensures every block has a unique fingerprint


5. Blockchain Class

In [14]:
class Blockchain:
    def __init__(self, difficulty: int = 3):
        self.chain = []
        self.difficulty = difficulty
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis = Block(index=0, timestamp=time.time(), data={"msg": "genesis"}, previous_hash="0")
        # Optionally mine the genesis block for consistency
        self.mine_block(genesis)
        self.chain.append(genesis)

    @property
    def last_block(self) -> Block:
        return self.chain[-1]

    def is_valid_proof(self, block: Block) -> bool:
        return block.hash.startswith("0" * self.difficulty)

    def mine_block(self, block: Block) -> Block:
        # Simple proof-of-work: find a nonce so hash has leading zeros
        while True:
            block.hash = block.compute_hash()
            if self.is_valid_proof(block):
                return block
            block.nonce += 1

    def add_block(self, data: dict) -> Block:
        new_block = Block(
            index=self.last_block.index + 1,
            timestamp=time.time(),
            data=data,
            previous_hash=self.last_block.hash
        )
        self.mine_block(new_block)
        # Integrity check before appending
        if new_block.previous_hash != self.last_block.hash or not self.is_valid_proof(new_block):
            raise ValueError("Invalid block—rejected.")
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self) -> bool:
        # Verify links and proof-of-work across the chain
        for i in range(1, len(self.chain)):
            curr = self.chain[i]
            prev = self.chain[i - 1]
            if curr.previous_hash != prev.hash:
                return False
            if curr.compute_hash() != curr.hash:
                return False
            if not self.is_valid_proof(curr):
                return False
        return True

Genesis Block

- The first block in the chain.
- Has no previous block, so previous_hash = "0".


Mining (Proof‑of‑Work)

- Mining means finding a hash that starts with a certain number of zeros (difficulty).
- The program keeps changing the nonce until the hash matches the rule.


Adding a Block

- Creates a new block with given data.
- Links it to the previous block.
- Mines it to meet difficulty.
- Adds it to the chain.


Validating the Chain

- Checks if:
- Each block correctly links to the previous one.
- Hashes are correct.
- Proof‑of‑work is valid.


6. Running the Program

In [17]:
if __name__ == "__main__":
    bc = Blockchain(difficulty=4)  # increase for slower mining
    bc.add_block({"amount": 10, "from": "alice", "to": "bob"})
    bc.add_block({"amount": 25, "from": "bob", "to": "carol"})

    for b in bc.chain:
        print(f"Index: {b.index}")
        print(f"Timestamp: {b.timestamp}")
        print(f"Data: {b.data}")
        print(f"PrevHash: {b.previous_hash}")
        print(f"Nonce: {b.nonce}")
        print(f"Hash: {b.hash}")
        print("-" * 60)

    print("Chain valid?", bc.is_chain_valid())

Index: 0
Timestamp: 1767759734.681307
Data: {'msg': 'genesis'}
PrevHash: 0
Nonce: 1768
Hash: 00008e775a4183f18dc6753a0ab8b60851f307a3b514d5df1cd4b8701eccc070
------------------------------------------------------------
Index: 1
Timestamp: 1767759734.6975646
Data: {'amount': 10, 'from': 'alice', 'to': 'bob'}
PrevHash: 00008e775a4183f18dc6753a0ab8b60851f307a3b514d5df1cd4b8701eccc070
Nonce: 31903
Hash: 000097cd4f03a4566b57c363bfcf3ac4596d1d6744d69dbdaf6a19e94363e315
------------------------------------------------------------
Index: 2
Timestamp: 1767759734.9799445
Data: {'amount': 25, 'from': 'bob', 'to': 'carol'}
PrevHash: 000097cd4f03a4566b57c363bfcf3ac4596d1d6744d69dbdaf6a19e94363e315
Nonce: 3828
Hash: 0000c8a8132dda9cd80d49372706b372e78955ced5750aa9c136dc591de75084
------------------------------------------------------------
Chain valid? True
